# 📊 Model Evaluation & Comparison

This notebook evaluates forecasting model performance and compares different approaches.

## Goals
- Evaluate model accuracy metrics (MAE, RMSE, MAPE)
- Compare different forecasting models
- Visualize predictions vs actuals
- Identify best performing models per product segment

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Libraries loaded")

✓ Libraries loaded


## 1. Load Test Data and Predictions

In [ ]:
# Load test data and predictions from notebook 03
# Make sure you ran 03_training.ipynb first!

# We'll recreate the test dataframe with predictions
# This assumes notebook 03 has been run and variables are available
try:
    # Check if variables from notebook 03 exist
    print(f"Test set size: {len(test)} rows")
    print(f"Predictions available: {len(y_pred)} predictions")
    
    # Create evaluation dataframe
    df_eval = test[['id_produit', 'date', 'quantite_demande']].copy()
    df_eval['actual'] = y_test.values
    df_eval['predicted'] = y_pred
    
    print("\n✓ Data loaded successfully!")
    print(f"Date range: {df_eval['date'].min()} to {df_eval['date'].max()}")
    print(f"Products: {df_eval['id_produit'].nunique()}")
    
    df_eval.head(10)
    
except NameError:
    print("❌ Error: Variables from notebook 03 not found!")
    print("Please run 03_training.ipynb first, then run this notebook.")

Load your test data with predictions here
Expected columns: date, product_id, actual, predicted_model1, predicted_model2, ...


## 2. Calculate Evaluation Metrics

In [ ]:
def calculate_metrics(y_true, y_pred):
    """
    Calculate common forecasting metrics
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # MAPE (Mean Absolute Percentage Error) - only for non-zero actuals
    mask = y_true != 0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan
    
    # R-squared
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R²': r2
    }

# Calculate overall metrics
metrics = calculate_metrics(df_eval['actual'], df_eval['predicted'])

print("="*60)
print("OVERALL METRICS")
print("="*60)
for metric, value in metrics.items():
    if metric == 'MAPE':
        print(f"{metric:10s}: {value:.2f}%")
    else:
        print(f"{metric:10s}: {value:.4f}")
print("="*60)

## 3. Compare Multiple Models

In [ ]:
# Compare LightGBM with simple baseline models

# Naive forecast: use yesterday's demand (lag_1)
df_eval['predicted_naive'] = test['lag_1'].values

# 7-day moving average
df_eval['predicted_ma7'] = test['rolling_mean_7'].values

# Compare all models
model_names = ['predicted', 'predicted_naive', 'predicted_ma7']
model_labels = ['LightGBM', 'Naive (Yesterday)', '7-Day MA']

comparison_results = []
for model_col, label in zip(model_names, model_labels):
    metrics = calculate_metrics(df_eval['actual'], df_eval[model_col])
    metrics['Model'] = label
    comparison_results.append(metrics)

df_comparison = pd.DataFrame(comparison_results)
df_comparison = df_comparison[['Model', 'MAE', 'RMSE', 'MAPE', 'R²']]

print("\n📊 MODEL COMPARISON")
print("="*80)
print(df_comparison.to_string(index=False))
print("="*80)
print("\n💡 Lower MAE/RMSE/MAPE is better | Higher R² is better")

## 4. Visualize Model Performance

In [ ]:
# Plot metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics_to_plot = ['MAE', 'RMSE', 'MAPE', 'R²']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    bars = ax.bar(df_comparison['Model'], df_comparison[metric], color=colors[idx], alpha=0.7)
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        label = f'{height:.2f}%' if metric == 'MAPE' else f'{height:.4f}'
        ax.text(bar.get_x() + bar.get_width()/2., height,
                label, ha='center', va='bottom', fontsize=10)
    
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.show()

print("\n✓ Best model highlighted by lowest MAE/RMSE/MAPE and highest R²")

## 5. Actual vs Predicted Plot

In [ ]:
# Actual vs Predicted scatter plot for LightGBM
plt.figure(figsize=(12, 6))

plt.scatter(df_eval['actual'], df_eval['predicted'], alpha=0.4, s=20, c='#4ECDC4', edgecolors='navy', linewidth=0.5)

# Perfect prediction line
max_val = max(df_eval['actual'].max(), df_eval['predicted'].max())
plt.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect Prediction', alpha=0.8)

plt.xlabel('Actual Demand', fontsize=12, fontweight='bold')
plt.ylabel('Predicted Demand', fontsize=12, fontweight='bold')
plt.title('LightGBM: Actual vs Predicted Demand', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Add statistics box
mae_val = calculate_metrics(df_eval['actual'], df_eval['predicted'])['MAE']
rmse_val = calculate_metrics(df_eval['actual'], df_eval['predicted'])['RMSE']
textstr = f'MAE: {mae_val:.2f}\nRMSE: {rmse_val:.2f}'
plt.text(0.05, 0.95, textstr, transform=plt.gca().transAxes, 
         fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 6. Time Series Forecast Visualization

In [ ]:
# Visualize forecast for a specific product
products_sorted = df_eval.groupby('id_produit')['actual'].sum().sort_values(ascending=False)
top_product = products_sorted.index[0]

df_product = df_eval[df_eval['id_produit'] == top_product].sort_values('date')

plt.figure(figsize=(16, 6))
plt.plot(df_product['date'], df_product['actual'], 'o-', label='Actual', linewidth=2.5, markersize=6, color='#2E4057')
plt.plot(df_product['date'], df_product['predicted'], 's--', label='LightGBM Predicted', alpha=0.8, linewidth=2, markersize=5, color='#FF6B6B')
plt.plot(df_product['date'], df_product['predicted_naive'], '^:', label='Naive (Yesterday)', alpha=0.6, linewidth=1.5, markersize=4, color='#95DAC1')

plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Demand Quantity', fontsize=12, fontweight='bold')
plt.title(f'Demand Forecast for Product {top_product} (Highest Volume Product)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n📦 Product {top_product} - Total actual demand in test period: {df_product['actual'].sum():.0f} units")

## 7. Error Distribution Analysis

In [ ]:
# Error distribution analysis
df_eval['error'] = df_eval['actual'] - df_eval['predicted']
df_eval['abs_error'] = np.abs(df_eval['error'])
df_eval['pct_error'] = np.where(df_eval['actual'] != 0, 
                                 (df_eval['error'] / df_eval['actual']) * 100, 
                                 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Error distribution
axes[0].hist(df_eval['error'], bins=50, edgecolor='black', color='#4ECDC4', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].axvline(df_eval['error'].mean(), color='orange', linestyle='--', linewidth=2, label=f'Mean: {df_eval["error"].mean():.2f}')
axes[0].set_xlabel('Prediction Error (Actual - Predicted)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Distribution of Prediction Errors', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Absolute error distribution
axes[1].hist(df_eval['abs_error'], bins=50, edgecolor='black', color='#FF6B6B', alpha=0.7)
axes[1].axvline(df_eval['abs_error'].mean(), color='darkred', linestyle='--', linewidth=2, label=f'Mean: {df_eval["abs_error"].mean():.2f}')
axes[1].set_xlabel('Absolute Error', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1].set_title('Distribution of Absolute Errors', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Percentage error distribution (clipped)
axes[2].hist(df_eval['pct_error'].clip(-100, 100), bins=50, edgecolor='black', color='#95DAC1', alpha=0.7)
axes[2].axvline(0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Percentage Error (%) [clipped ±100%]', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[2].set_title('Distribution of Percentage Errors', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Error Statistics:")
print(f"Mean Error: {df_eval['error'].mean():.2f}")
print(f"Median Error: {df_eval['error'].median():.2f}")
print(f"Std Dev: {df_eval['error'].std():.2f}")
print(f"Mean Absolute Error: {df_eval['abs_error'].mean():.2f}")

## 8. Model Performance by Product Category

In [ ]:
# Performance by product - top 10 and bottom 10
product_performance = df_eval.groupby('id_produit').apply(
    lambda x: pd.Series({
        'MAE': mean_absolute_error(x['actual'], x['predicted']),
        'RMSE': np.sqrt(mean_squared_error(x['actual'], x['predicted'])),
        'Total_Demand': x['actual'].sum(),
        'Count': len(x)
    })
).reset_index()

print("="*80)
print("TOP 10 PRODUCTS - LOWEST MAE (Best Predictions)")
print("="*80)
print(product_performance.nsmallest(10, 'MAE').to_string(index=False))

print("\n" + "="*80)
print("BOTTOM 10 PRODUCTS - HIGHEST MAE (Worst Predictions)")
print("="*80)
print(product_performance.nlargest(10, 'MAE').to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_10 = product_performance.nsmallest(10, 'MAE')
axes[0].barh(top_10['id_produit'].astype(str), top_10['MAE'], color='#96CEB4', alpha=0.7)
axes[0].set_xlabel('MAE', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Product ID', fontsize=11, fontweight='bold')
axes[0].set_title('Top 10 Products - Best Predictions (Lowest MAE)', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

bottom_10 = product_performance.nlargest(10, 'MAE')
axes[1].barh(bottom_10['id_produit'].astype(str), bottom_10['MAE'], color='#FF6B6B', alpha=0.7)
axes[1].set_xlabel('MAE', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Product ID', fontsize=11, fontweight='bold')
axes[1].set_title('Bottom 10 Products - Worst Predictions (Highest MAE)', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary & Conclusions

In [ ]:
print("\n" + "="*80)
print("📊 MODEL EVALUATION SUMMARY")
print("="*80)

# Best model
best_model_row = df_comparison.loc[df_comparison['MAE'].idxmin()]
print(f"\n🏆 Best Model (by MAE): {best_model_row['Model']}")
print(f"   MAE:  {best_model_row['MAE']:.4f}")
print(f"   RMSE: {best_model_row['RMSE']:.4f}")
print(f"   MAPE: {best_model_row['MAPE']:.2f}%")
print(f"   R²:   {best_model_row['R²']:.4f}")

# Performance improvement over naive
naive_mae = df_comparison[df_comparison['Model'] == 'Naive (Yesterday)']['MAE'].values[0]
lgb_mae = df_comparison[df_comparison['Model'] == 'LightGBM']['MAE'].values[0]
improvement = ((naive_mae - lgb_mae) / naive_mae) * 100

print(f"\n📈 LightGBM Improvement over Naive Baseline:")
print(f"   MAE reduced by: {improvement:.1f}%")

# Error characteristics
print(f"\n📉 Error Characteristics:")
print(f"   Mean error: {df_eval['error'].mean():.2f} (bias)")
print(f"   Median error: {df_eval['error'].median():.2f}")
print(f"   Error std dev: {df_eval['error'].std():.2f}")
print(f"   Mean absolute error: {df_eval['abs_error'].mean():.2f}")

# Accuracy buckets
within_5 = (df_eval['abs_error'] <= 5).sum() / len(df_eval) * 100
within_10 = (df_eval['abs_error'] <= 10).sum() / len(df_eval) * 100
within_20 = (df_eval['abs_error'] <= 20).sum() / len(df_eval) * 100

print(f"\n🎯 Prediction Accuracy Buckets:")
print(f"   Within ±5 units:  {within_5:.1f}%")
print(f"   Within ±10 units: {within_10:.1f}%")
print(f"   Within ±20 units: {within_20:.1f}%")

print("\n" + "="*80)
print("💡 NEXT STEPS")
print("="*80)
print("  1. ✓ LightGBM significantly outperforms simple baselines")
print("  2. → Fine-tune hyperparameters for further improvement")
print("  3. → Implement ensemble methods (combine multiple models)")
print("  4. → Deploy best model to production API")
print("  5. → Set up monitoring and automatic retraining pipeline")
print("  6. → Handle products with high error separately (different strategy)")
print("="*80)


MODEL EVALUATION SUMMARY

💡 Next Steps:
  1. Fine-tune hyperparameters of best model
  2. Implement ensemble methods
  3. Deploy model to production API
  4. Set up monitoring and retraining pipeline


---

## 📝 Notes

Add your observations and insights here:
- Which model performed best?
- Are there any patterns in the errors?
- Which product categories are hardest to forecast?
- What improvements could be made?